# Outlier Analysis — Detección de anomalías en datos crudos

> Notebook de análisis de outliers para ejecutar **antes del ETL**. Detecta valores extremos, errores de medición y patrones anómalos en el consumo que pueden distorsionar el entrenamiento si no se tratan a tiempo.

## Overview

Este notebook aplica tres métodos estadísticos complementarios para detectar outliers y produce recomendaciones accionables de preprocesamiento:

- **IQR (Interquartile Range)**: Robusto a valores extremos. Marca como outlier todo valor fuera de $[Q_1 - 1.5 \times IQR,\; Q_3 + 1.5 \times IQR]$.
- **Z-Score**: Desviaciones estándar respecto a la media. Útil cuando la distribución es aproximadamente normal. Umbral típico: $|z| > 3$.
- **Modified Z-Score**: Basado en MAD (*Median Absolute Deviation*), resistente a outliers. Preferible cuando los datos son asimétricos o ya tienen outliers extremos.

## Cuándo usar este notebook

- **Antes del ETL**: Identificar problemas de calidad de datos temprano
- **Durante el EDA**: Entender distribuciones y anomalías
- **Al detectar drift**: Comparar patrones de outliers entre períodos

Consultá `outlier_analysis.md` para instrucciones detalladas.

## 1. Setup — Configuración inicial

Configurá la ruta al dataset y la columna target. El notebook detecta automáticamente si está corriendo dentro del repositorio del framework o en un proyecto generado con `energizados init`.

**Variables clave:**
- `DATASET_PATH`: ruta al archivo Parquet con los datos a analizar
- `TARGET_COL`: nombre de la columna target (opcional — usar `None` si no hay target)

In [ ]:
# outlier_analysis.ipynb
# Outlier Analysis — Template Notebook
# ====================================
# Use this notebook to investigate outliers in the training dataset.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import HTML, display
from energizados.eda._outlier_detector import OutlierDetector
from energizados.eda.utils import classify_columns
from energizados.eda.plots import EDAStaticPlots

# Configuration
# ============================================================
# ⚠️  CONFIGURACIÓN: todos los paths y nombres de archivos se definen acá
# ============================================================
PROJECT_PATH = Path("/home/vvv/Develop/bid/energizados/.proyects/celesc")
RAW_PATH = PROJECT_PATH / "data/raw/v2"
PROC_PATH = PROJECT_PATH / "data/processed/v5"
REPORT_DIR = Path(".")

# === Nombres de archivos ===
DATASET_FILE = "dataset_celesc_train.parquet"
REPORT_FILE = "outlier_report.json"

# === Paths derivados ===
DATASET_PATH = PROC_PATH / DATASET_FILE
REPORT_PATH = REPORT_DIR / REPORT_FILE
TARGET_COL = "target"

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Plotter instance for EDA static plots (output_dir only used for saving, not for SVG generation)
plotter = EDAStaticPlots(output_dir=".")

print("✅ Setup complete")


## 2. Carga de datos

Cargamos el dataset y mostramos información básica: dimensiones, tipos de datos, valores nulos y estadísticas descriptivas. Esto nos da una primera impresión de la calidad y estructura de los datos antes de aplicar cualquier método de detección.

In [ ]:
# Load dataset
df = pd.read_parquet(DATASET_PATH)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

# Basic info
print("\n--- Dataset Info ---")
df.info()

# First few rows
print("\n--- First 5 Rows ---")
display(df.head())

## 3. Clasificación de columnas

Clasificamos automáticamente las columnas en tres tipos:

- **Numéricas**: variables continuas (ej. montos, edades)
- **Categóricas**: variables discretas con pocos valores únicos
- **Consumo**: columnas que siguen el patrón `*_anterior` (serie temporal de consumo mensual)

Esta clasificación es importante porque cada tipo requiere un tratamiento distinto: los outliers en consumo tienen implicancias diferentes a los outliers en variables administrativas.

In [ ]:
# Classify columns automatically, excluding identifiers and geographic coordinates.
col_types = classify_columns(
    df,
    id_col="cliente",
    lat_col="latitude",
    lon_col="longitude",
    date_col="periodo",
)
numeric_cols = [c for c in col_types.get("numeric", []) if c != "geo_cluster"]
categorical_cols = col_types.get("categorical", [])
consumption_cols = col_types.get("consumption", [])
analysis_cols = numeric_cols + consumption_cols

print("📊 Column Classification:")
print(f"  - Numeric: {len(numeric_cols)} columns")
print(f"  - Categorical: {len(categorical_cols)} columns")
print(f"  - Consumption: {len(consumption_cols)} columns")

if numeric_cols:
    print(f"\nNumeric columns: {numeric_cols}")
if consumption_cols:
    print(f"\nConsumption columns: {consumption_cols}")


## 4. Detección multi-método

Aplicamos IQR, Z-Score y Modified Z-Score **en paralelo** sobre todas las columnas numéricas y de consumo. Usar múltiples métodos nos da una visión más robusta:

- Un valor marcado por los 3 métodos es casi seguro un outlier real
- Un valor marcado solo por Z-Score pero no por IQR podría ser un falso positivo en datos asimétricos
- El Modified Z-Score es el más confiable cuando la distribución ya tiene outliers extremos

El resultado es un diccionario `{columna: {método: [índices]}}` que usaremos para construir la tabla resumen.

In [ ]:
# Detect outliers using multiple methods.
detector = OutlierDetector(
    methods=["iqr", "zscore", "modified_zscore"],
    iqr_multiplier=1.5,
    zscore_threshold=3.0,
    store_mask=True,
)

outlier_summary = {}
for col in analysis_cols:
    result = detector.detect(df[col])
    for method_result in result.values():
        if "mask" in method_result:
            method_result["mask"] = method_result["mask"].reindex(
                df.index, fill_value=False
            )
    outlier_summary[col] = result
    iqr_result = result.get("iqr", {})
    print(f"✓ {col}: IQR outliers = {iqr_result.get('outlier_pct', 0.0):.2f}%")

print(
    f"\n📈 Detected outliers for {len(analysis_cols)} "
    "numeric and consumption columns"
)


## 5. Tabla resumen

Comparamos los conteos de outliers detectados por cada método. Esta tabla permite identificar rápidamente:

- **Columnas con muchos outliers**: posibles errores sistemáticos de medición
- **Métodos que divergen**: cuando un método detecta muchos más outliers que los otros, puede indicar que ese método no es apropiado para la distribución de esa columna
- **Columnas sin outliers**: datos limpios que no requieren intervención

In [ ]:
# Build summary table.
summary_rows = []
for col, result in outlier_summary.items():
    iqr_result = result.get("iqr", {})
    zscore_result = result.get("zscore", {})
    summary_rows.append(
        {
            "column": col,
            "iqr_count": iqr_result.get("outlier_count", 0),
            "iqr_pct": iqr_result.get("outlier_pct", 0.0),
            "zscore_count": zscore_result.get("outlier_count", 0),
            "zscore_pct": zscore_result.get("outlier_pct", 0.0),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("iqr_pct", ascending=False)

print("=== Outlier Summary Table ===")
print(summary_df.to_string(index=False))

display(summary_df.style.background_gradient(subset=["iqr_pct"], cmap="Reds"))


## 6. Boxplots — Visualización de outliers

Generamos boxplots solo para las columnas que tienen al menos un outlier detectado. Los puntos fuera de los *bigotes* (whiskers) son los valores marcados como outliers por el método IQR.

**Qué observar:**
- Outliers **unilaterales** (todos hacia arriba o todos hacia abajo): posible error de medición o censura
- Outliers **extremos** (muy alejados de los bigotes): probablemente errores de carga de datos
- Outliers **moderados** y dispersos: podrían ser casos legítimos pero inusuales

In [ ]:
# Boxplots with outlier markers.
cols_with_outliers = [
    col
    for col, result in outlier_summary.items()
    if result.get("iqr", {}).get("outlier_pct", 0) > 0
]

if cols_with_outliers:
    print(f"📊 Creating boxplots for {len(cols_with_outliers)} columns with outliers...")

    outlier_masks = {
        col: outlier_summary[col]["iqr"].get(
            "mask", pd.Series(False, index=df.index)
        )
        for col in cols_with_outliers
    }
    svg_dict = plotter.plot_outlier_boxplots(
        df, cols_with_outliers[:6], outlier_masks
    )

    for plot_name, svg in list(svg_dict.items())[:3]:
        print(f"\n=== {plot_name} ===")
        display(HTML(svg))
else:
    print("✅ No columns with outliers detected")


## 7. Anomalías en consumo

Las columnas de consumo mensual (`*_anterior`) merecen un análisis específico porque son la señal principal para la detección de fraude:

- **Consumo cero persistente**: posible medidor detenido o puenteado — señal clásica de fraude
- **Picos aislados**: un mes con consumo 10× el promedio puede ser error de lectura, no fraude
- **Caídas abruptas**: una baja drástica y sostenida puede indicar intervención del medidor

Este análisis complementa al detector de outliers genérico con una mirada específica del dominio energético.

In [ ]:
# Analyze consumption-specific patterns
if consumption_cols:
    print(f'📊 Analyzing {len(consumption_cols)} consumption columns...')
    
    # Build a combined outlier mask across all consumption columns (optional)
    combo_mask = pd.Series(False, index=df.index)
    for col in consumption_cols:
        if col in outlier_summary:
            col_mask = outlier_summary[col].get('iqr', {}).get('mask', pd.Series(False, index=df.index))
            combo_mask = combo_mask | col_mask
    
    # plot_consumption_anomalies signature:
    #   (df, consumption_cols, outlier_mask=None, target_col=None)
    svg_dict = plotter.plot_consumption_anomalies(
        df, consumption_cols,
        outlier_mask=combo_mask if combo_mask.any() else None,
        target_col=TARGET_COL if TARGET_COL and TARGET_COL in df.columns else None,
    )
    
    # Display first few periods
    for period, svg in list(svg_dict.items())[:3]:
        print(f'\n=== {period} ===')
        display(HTML(svg))
else:
    print("ℹ️  No consumption columns detected")


## 8. Análisis a nivel de fila

Identificamos filas que son outliers en **múltiples columnas simultáneamente**. Un cliente con valores extremos en 5 de 12 meses de consumo es mucho más sospechoso que uno con un outlier aislado en un solo mes.

**Métrica clave:** *outlier score* por fila = cantidad de columnas donde esa fila fue marcada. Ordenamos de mayor a menor para priorizar la investigación.

In [ ]:
# Which rows are outliers across multiple columns?
outlier_counts_per_row = pd.Series(0, index=df.index, dtype="int64")
for col in analysis_cols:
    mask = outlier_summary[col].get("iqr", {}).get(
        "mask", pd.Series(False, index=df.index)
    )
    outlier_counts_per_row += mask.reindex(df.index, fill_value=False).astype(int)

df["outlier_score"] = outlier_counts_per_row

print("=== Outlier Score Distribution ===")
print(df["outlier_score"].describe())

top_outliers = df.nlargest(10, "outlier_score")
print(f"\nTop 10 outlier-prone rows (out of {len(df)} total):")

display_cols = ["outlier_score"] + analysis_cols[:5]
if TARGET_COL and TARGET_COL in df.columns:
    display_cols.append(TARGET_COL)

display(top_outliers[display_cols])


## 9. Recomendaciones de preprocesamiento

Basado en los patrones de outliers detectados, el notebook sugiere acciones concretas:

- **Winsorización (capping)**: limitar valores extremos al percentil 1 y 99. Recomendado cuando hay pocos outliers extremos que parecen errores de medición.
- **Transformación logarítmica**: comprimir la escala cuando los datos tienen cola larga. Útil para variables de consumo que naturalmente varían en órdenes de magnitud.
- **Eliminación de filas**: solo cuando una fila es outlier en muchas columnas y hay evidencia de que es un error sistémico, no un caso legítimo.
- **Imputación**: reemplazar outliers por la mediana o media recortada del grupo.

La recomendación incluye qué columnas tratar y con qué método.

In [ ]:
# Print preprocessing recommendations.
print("=== PREPROCESSING RECOMMENDATIONS ===\n")

for col, result in outlier_summary.items():
    pct = result.get("iqr", {}).get("outlier_pct", 0.0)

    if pct > 20:
        status = "⚠️ HIGH — Consider capping (winsorizing) or removing column"
        action = "winsorize"
    elif pct > 10:
        status = "🔍 MEDIUM — Investigate origin before deciding"
        action = "investigate"
    elif pct > 5:
        status = "🔍 LOW-MEDIUM — Monitor in next data refresh"
        action = "monitor"
    else:
        status = "✅ OK — Within acceptable range"
        action = "none"

    print(f"{col}: {pct:.1f}% outliers — {status}")
    print(f"   Recommended action: {action}\n")

high_outlier_cols = [
    col
    for col, result in outlier_summary.items()
    if result.get("iqr", {}).get("outlier_pct", 0.0) > 20
]
if high_outlier_cols:
    print(f"\n⚠️  {len(high_outlier_cols)} columns with high outlier percentage:")
    print(f"   {high_outlier_cols}")


## 10. Exportar reporte

Guardamos los resultados del análisis como JSON para trazabilidad y documentación. Esto permite:

- Comparar análisis de outliers entre diferentes versiones del dataset
- Auditar decisiones de preprocesamiento
- Compartir hallazgos con el equipo sin necesidad de re-ejecutar el notebook

In [ ]:
# Export outlier report as JSON.
import json
from datetime import datetime


def strip_masks(value):
    if isinstance(value, dict):
        return {
            key: strip_masks(item)
            for key, item in value.items()
            if key not in {"mask", "outlier_mask"}
        }
    if isinstance(value, list):
        return [strip_masks(item) for item in value]
    return value


report = {
    "generated_at": str(datetime.now()),
    "dataset": str(DATASET_PATH),
    "n_rows": len(df),
    "n_numeric_cols": len(numeric_cols),
    "n_categorical_cols": len(categorical_cols),
    "n_consumption_cols": len(consumption_cols),
    "n_analyzed_cols": len(analysis_cols),
    "target_column": TARGET_COL,
    "outlier_summary": {
        col: strip_masks(result)
        for col, result in outlier_summary.items()
    },
}

output_file = REPORT_PATH
with open(output_file, "w") as f:
    json.dump(report, f, indent=2, default=str)

print(f"✅ Report saved to {output_file}")
print(f"   {len(outlier_summary)} columns analyzed")

import os
file_size = os.path.getsize(output_file)
print(f"   File size: {file_size / 1024:.1f} KB")


## 11. Próximos pasos

Después de completar este análisis:

1. **Revisar columnas con alto porcentaje de outliers**: investigar si son errores de carga, mediciones defectuosas o casos legítimos pero extremos
2. **Aplicar preprocesamiento**: winsorización, capping o transformaciones según las recomendaciones
3. **Actualizar el pipeline ETL**: agregar los pasos de limpieza identificados en `config/etl.yaml`
   (ej. usar `ClipOutliersETL` para columnas de consumo)
4. **Documentar decisiones**: registrar qué se hizo y por qué para auditoría y reproducibilidad
5. **Re-ejecutar el análisis**: después del preprocesamiento, volver a correr este notebook para validar que los outliers se redujeron a niveles aceptables

---
### Recursos relacionados

- `outlier_analysis.md` — instrucciones detalladas y guía de interpretación
- `energizados.eda._outlier_detector.OutlierDetector` — implementación de los métodos
- `energizados.etl.pipeline.ClipOutliersETL` — ETL para recortar outliers en el pipeline
- `notebooks/01_data_check.ipynb` — verificación de calidad de datos complementaria